# Chương 10. Định hình lại và kết hợp dữ liệu

**Câu hỏi mở đầu:** Nếu thông tin nằm ở nhiều bảng khác nhau, làm thế nào tạo được một bảng phân tích đáng tin cậy?

Notebook này là tài nguyên đồng hành của chương. Mỗi phần đều đi theo nhịp **câu hỏi → dữ liệu → mã → kết quả → diễn giải → kiểm tra bằng chứng**.

## Mục tiêu

- Tái hiện các ví dụ cốt lõi của chương bằng mã có thể chạy lại.
- Kiểm tra giả định trước khi diễn giải output.
- Kết thúc bằng ít nhất một câu hỏi về điều mà kết quả **chưa** cho biết.

In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("error", category=FutureWarning)
warnings.filterwarnings("error", category=DeprecationWarning)
ROOT = Path.cwd()
DATA = ROOT / "data"
print("Working root:", ROOT)


In [ ]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", 20)


## 1. Khóa và quan hệ giữa bảng

In [ ]:
tx=pd.read_csv(DATA / "metromart" / "metromart_transactions.csv")
customers=pd.read_csv(DATA / "metromart" / "metromart_customers.csv")
products=pd.read_csv(DATA / "metromart" / "metromart_products.csv")
loyalty=pd.read_csv(DATA / "metromart" / "metromart_loyalty_history.csv")
print(len(tx), customers["customer_id"].is_unique, products["product_id"].is_unique)
print("loyalty customer unique?", loyalty["customer_id"].is_unique)

## 2. Phép nối nhiều–một hợp lệ

In [ ]:
joined=tx.merge(customers,on="customer_id",how="left",validate="many_to_one")
joined=joined.merge(products,on="product_id",how="left",validate="many_to_one",suffixes=("","_product"))
print("rows before/after:", len(tx), len(joined))

## 3. Signature example: 12.000 → 38.400

In [ ]:
naive=tx.merge(loyalty,on="customer_id",how="left")
print("before:",len(tx),"after:",len(naive),"factor:",len(naive)/len(tx))

Python không sai. Câu hỏi sai là giả định “chỉ cần cùng customer_id là đủ”. Lịch sử hạng thành viên có nhiều giai đoạn hiệu lực cho cùng khách hàng.

## 4. Điều kiện thời gian

In [ ]:
tx2=tx.copy()
tx2["transaction_date"]=pd.to_datetime(tx2["transaction_date"])
loy=loyalty.copy()
loy["valid_from"]=pd.to_datetime(loy["valid_from"])
loy["valid_to"]=pd.to_datetime(loy["valid_to"])
probe=tx2.iloc[0]
candidates=loy[(loy["customer_id"]==probe["customer_id"]) &
               (loy["valid_from"]<=probe["transaction_date"]) &
               (probe["transaction_date"]<=loy["valid_to"])]
print("candidate membership rows for first transaction:", len(candidates))
display(candidates)

### Kiểm tra bằng chứng

Trước khi tin một con số sau merge, kiểm tra khóa, quan hệ, số hàng và đơn vị phân tích.

## Thực hành

Tạo một bảng phân tích từ transactions + customers + products và chứng minh bằng code rằng số hàng không tăng ngoài dự kiến.

---
### Bạn đã sẵn sàng sang chương tiếp theo nếu có thể…

- giải thích output bằng lời;
- chỉ ra ít nhất một giả định;
- nói được kết quả chưa cho phép kết luận điều gì.

**Exit check:** Nếu mã chạy không lỗi nhưng câu trả lời trái với ý nghĩa của dữ liệu, bạn sẽ kiểm tra điều gì trước?